In [1]:
import os
os.chdir('..')

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from umap import UMAP
from sklearn.cluster import HDBSCAN
from src.objectives import Objective
from numba import njit

In [ ]:
# Loads checkpoint to visualise
CHECKPOINT_PATH = 'saved_models/visual-maddist-montezumas-revenge.pt'
checkpoint = Objective.load_checkpoint(CHECKPOINT_PATH)

In [ ]:
# Embeds observations from the dataset (every 1 in 15 frames)
FRAME_SKIP = 15

obs = checkpoint.dataset.observations
strided_obs = obs[::FRAME_SKIP]
indices = np.arange(obs.shape[0])

embeddings = checkpoint.encode(torch.from_numpy(strided_obs)).numpy()

In [ ]:
# JIT definition of the MadDist simple quasimetric function for UMAP
@njit(fastmath=True)
def maddist_distance_numba(u, v):
    alpha = 0.5
    
    diff = u - v
    
    relus = np.maximum(0.0, diff)
    
    max_val = np.max(relus)
    mean_val = np.mean(relus)
    
    return alpha * max_val + (1.0 - alpha) * mean_val

In [ ]:
# Uses UMAP to reduce embeddings to 2 dimensions
reducer = UMAP(
    n_neighbors=30,
    min_dist=0.2,
    metric=maddist_distance_numba,
    random_state=42
)

embeddings_2d = reducer.fit_transform(embeddings)

In [ ]:
# Uses HDBSCAN to cluster states in the reduced embedding space
clusterer = HDBSCAN(
    min_cluster_size=200,
    min_samples=15, 
    cluster_selection_epsilon=0.0, 
)

cluster_labels = clusterer.fit_predict(embeddings_2d)

In [ ]:
# Chooses valid (non-outlier) points for plotting
valid_clusters_mask = cluster_labels != -1
clean_embeddings_2d = embeddings_2d[valid_clusters_mask]
clean_labels = cluster_labels[valid_clusters_mask]
clean_indices = indices[valid_clusters_mask]

# Generates cluster colour map
unique_clusters = np.unique(clean_labels)
n_clusters = len(unique_clusters)
base_cmap = plt.get_cmap('turbo')
colors = base_cmap(np.linspace(0, 1, n_clusters))

# Adjusts colour map to not include colours that are too bright against the white background
for i in range(len(colors)):
    r, g, b, a = colors[i]
    
    luminance = 0.299 * r + 0.587 * g + 0.114 * b
    
    max_allowed_luminance = 0.5

    if luminance > max_allowed_luminance:
        darken_factor = max_allowed_luminance / luminance
        colors[i, 0] *= darken_factor
        colors[i, 1] *= darken_factor
        colors[i, 2] *= darken_factor
        
# Shuffles the colours randomly to reduce nearby clusters having similar colours
np.random.seed(42)
np.random.shuffle(colors)

cmap = ListedColormap(colors)

In [ ]:
%matplotlib inline

fig, ax = plt.subplots(figsize=(8, 8))
scatter_plot = ax.scatter(
    clean_embeddings_2d[:, 0],
    clean_embeddings_2d[:, 1],
    s=1,
    alpha=1,
    c=clean_labels,
    cmap=cmap,
    picker=True,
)
ax.set_title("Montezuma's Revenge Latent Space Clusters", fontsize=16)
ax.axis('off')

plt.show()